In [2]:
from bs4 import BeautifulSoup
import copy
import json
import os
from PIL import Image

In [67]:
class Line:
    def __init__(self, boxes: list):
        self.boxes = boxes

    def __len__(self):
        return len(self.boxes)

    def __iter__(self):
        return iter(self.boxes)

    def __getitem__(self, idx: int):
        return self.boxes[idx]

    def __delitem__(self, idx: int):
        del self.boxes[idx]
        
class TextLine(Line):
    def __repr__(self):
        return f"TextLine({"".join([box["text_content"] for box in self.boxes])})"

class LyricsLine(Line):
    def __repr__(self):
        return f"LyricsLine({"".join([box["text_content"] for box in self.boxes])})"

class MusicLine(Line):
    def __repr__(self):
        return f"MusicLine({"".join(["*" for box in self.boxes])})"

In [79]:
def determine_page_from_x_coordinate(images, x):
    offset = 0
    for page_idx, page in enumerate(images[::-1]):  # pages are read from right to left
        offset += image.width
        if x < offset:
            return len(images)-1-page_idx

def segment_boxes_into_lines(boxes: list):
    # Raw lines treat music lines (consisting of one notation line and
    # one lyrics line) as one line.
    raw_lines = []
    for box in boxes:
        if len(raw_lines) == 0 or raw_lines[-1][-1]["is_line_break"]:
            raw_lines.append([box])
        else:
            raw_lines[-1].append(box)

    lines = []
    for line in raw_lines:
        line_types = set([box["box_type"] for box in line])
        if len(line_types) == 1 and "Music" in line_types:
            if line[0]["notation_content"] is not None:
                lines.append(MusicLine(line))
            if line[0]["text_content"] is not None:
                lines.append(LyricsLine(line))
        elif "Music" not in line_types:
            lines.append(TextLine(line))
        else:
            print("WARNING:", line_types)
            print("".join([box["text_content"] for box in line]))
            
    return lines

        
def segment_into_blocks(boxes):
    """
    Group boxes of same box type together. We need this so we can choose the boundaries of the
    TEI components correctly, e.g. <head> enclosing the title boxes, etc.

    Also, the music boxes contain two positions each: The right side contains the musical notation,
    while the left side contains the lyric character. To correctly encode this, we need to split the
    data accordingly.

    For this, we need column break information to determine the position of each box correctly!
    """
    boxes = copy.deepcopy(boxes)
    blocks = []
    for box in boxes:
        # i)   If there are no blocks yet, make a new block.
        # ii)  If there is a block but the current box' type is different
        #      than the current block's type, make a new block.
        if len(blocks) and blocks[-1][-1]["box_type"] == box["box_type"]:
            blocks[-1].append(box)
        else:
            blocks.append([box])
    return blocks
    
def add_block_tags(blocks):
    blocks = copy.deepcopy(blocks)
    for block in blocks:
        block_type = block[0]["box_type"]
        if block_type == "Title":  # titles are <head>
            block[0]["text_content"] = "<head><title>" + block[0]["text_content"]
            block[-1]["text_content"] += "</title></head>"
        elif block_type == "Preface":
            block[0]["text_content"] = '<p ana="preface">' + block[0]["text_content"]
            block[-1]["text_content"] += "</p>"
        elif block_type == "Mode":
            block[0]["text_content"] = '<p ana="mode" rend="small">' + block[0]["text_content"]
            block[-1]["text_content"] += "</p>"
        elif block_type == "Music":
            block[0]["text_content"] = '<lg><l>' + block[0]["text_content"]
            block[-1]["text_content"] += "</l></lg>"
    return blocks

In [80]:
class JsonBased:
    def __init__(self, file_path: str):
        with open(file_path, "r") as file_handle:
            self.json = json.load(file_handle)
            self.boxes = self.json["content"]
            self.image_files = self.json["images"]
            self.file = os.path.basename(file_path)
            self.physical_lines = segment_boxes_into_lines(self.boxes)

class Song(JsonBased):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.title = "".join([box["text_content"] for box in self.boxes if box["box_type"] == "Title"])

    def __str__(self):
        return f"Song({self.title}, {self.file})"
    
    def to_tei(self):
        tei_string = ""
        for line in self.lines:
            tei_string += "".join([box["text_content"] for box in line]) + "\n"

        return f'<div type="song">{tei_string}</div>'

class SongOfYue(JsonBased):
    def __init__(self, file_path: str):
        with open(file_path, "r") as file_handle:
            self.json = json.load(file_handle)
            self.boxes = self.json["content"]
            # Songs of Yue have special structure regarding their content.
            # The title (plus an indication that the piece is to the right)
            # is found after the music and lyrics!
            title_boxes = [box for box in self.boxes if box["box_type"] in ("Unmarked", "Title")]
            music_boxes = [box for box in self.boxes if box["box_type"] not in ("Unmarked", "Title")]
            self.boxes = music_boxes + title_boxes
            self.image_files = self.json["images"]
            self.file = os.path.basename(file_path)
            self.physical_lines = segment_boxes_into_lines(self.boxes)

            # Also, there are additional page break boxes that should be removed
            for line in self.physical_lines:
                if line[-1]["text_content"] in (None, "", " "):
                    del line[-1]

            # Update boxes accordingly
            self.boxes = [box for line in self.physical_lines for box in line]

            # Mark the boxes according to their physical line properties

    def __str__(self):
        return f"SongOfYue({self.title}, {self.file})"

#print(Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/01_shangdiming.json").physical_lines)
print(Song(edition_directory + "./json/02_qinquyishou/015_guyuan.json").physical_lines)
#print(SongOfYue(edition_directory + "./json/03_yuejiuge/016_dishunchudiao.json").physical_lines)
#print(Song(edition_directory + "./json/04_ling/026_xiaochongshanling.json").physical_lines)
#print(Song(edition_directory + "./json/05_man/059_nishangzhongxu.json").physical_lines)
#print(Song(edition_directory + "./json/06_ziduqu/079_yangzhouman.json").physical_lines)
#print(Song(edition_directory + "./json/07_zizhiqu/088_quixiaoyin.json").physical_lines)
#print(Song(edition_directory + "./json/08_bieji/092_xiaochongshanling.json").physical_lines)

泛聲日暮四山𠔃烟霧暗前浦將維舟𠔃無所追我前𠔃不
[TextLine(古怨), MusicLine(************), LyricsLine(逮懷後來𠔃何處屢囬顧  ), MusicLine(*), MusicLine(*********************), MusicLine(*********), MusicLine(*********************), LyricsLine(世事𠔃何據手翻覆𠔃雲雨過金谷𠔃花謝委塵土悲), MusicLine(*********************), LyricsLine(佳人𠔃薄命誰為主豈不猶有春𠔃妾自傷𠔃遲暮髮), MusicLine(***), LyricsLine(將素 ), MusicLine(*********************), LyricsLine(歡有窮𠔃恨無數𢎺欲絶𠔃聲苦滿目江山𠔃淚沾), MusicLine(****************), LyricsLine(君不見年年汾水上𠔃 惟秋鴈飛去 )]


In [45]:

class Table:
    def __init__(self, file_path: str):
        pass


class Description:
    def __init__(self, file_path: str):
        pass


class Section:
    def __init__(self, information_file: str, children: list[Description | Table | Song]):
        with open(information_file, "r") as file_handle:
            self.information_json = json.load(file_handle)
            self.information_boxes = self.information_json["content"]
            self.information_boxes = [box for box in self.information_boxes if box["box_type"] in ("Title", "Preface")]

        self.children = children
        
        self.physical_lines = segment_boxes_into_lines(self.information_boxes)
        for child in children:
            self.physical_lines += child.lines
            
        self.image_files = set(self.information_json["images"] +
                               [file for child in self.children for file in child.image_files])
        self.boxes = (self.information_boxes + [box for child in self.children for box in child.boxes])
        self.file = os.path.basename(information_file)
        self.title = "".join([box["text_content"] for box in self.information_boxes if box["box_type"] == "Title"])
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Section({self.title}, {self.file})\n"
        for child in self.children:
            return_string += f"    {child}\n"
        return_string = return_string[:-1]
        return return_string
    
    def to_tei(self):
        preface_boxes = [box for box in self.information_boxes if box["box_type"] == "Preface"]

        tei_string = ""
        if len(preface_boxes):
            tei_string += '<p ana="preface">'
            for line in segment_boxes_into_lines(preface_boxes):
                preface_line = "".join([box["text_content"] for box in line])
                tei_string += f'<lb/>{preface_line}'
            tei_string += "</p>"
        
        for child in self.children:
            tei_string += child.to_tei()
        return f'<div type="section"><head><lb/>{self.title}</head>{tei_string}</div>'

    
class Juan:
    def __init__(self, information_file: str, children: list[Section]):
        with open(information_file, "r") as file_handle:
            self.information_json = json.load(file_handle)
            self.information_boxes = self.information_json["content"]
            self.information_boxes = [box for box in self.information_boxes if box["box_type"] == "Unmarked"]

        self.children = children

        self.physical_lines = segment_boxes_into_lines(self.information_boxes)
        for child in children:
            self.physical_lines += child.lines
            
        self.image_files = set(self.information_json["images"] +
                               [file for child in self.children for file in child.image_files])
        self.boxes = (self.information_boxes + [box for child in self.children for box in child.boxes])
        self.title = "".join([box["text_content"] for box in self.information_boxes])
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Juan({self.title})\n"
        for child in self.children:
            child_string = str(child).split("\n")
            child_string = "".join(["    "+string+"\n" for string in child_string])
            return_string += f"{child_string}"
        return_string = return_string[:-1]
        return return_string
    
    def to_tei(self):
        tei_string = ""
        for child in self.children:
            tei_string += child.to_tei()
        # Juan always start with page break
        return f'<div type="juan"><head><pb/><lb/>{self.title}</head>{tei_string}</div>'

class TableOfContents:
    def __init__(self, file_path: str):
        pass


class Book:
    def __init__(self, children: list[TableOfContents | Juan | Section]):
        self.children = children

        self.physical_lines = []
        for child in children:
            self.physical_lines += child.lines
            
        self.image_files = set([file for child in self.children for file in child.image_files])
        self.boxes = [box for child in self.children for box in child.boxes]
        
        # assign page data to boxes
        
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Book(\n"
        for child in self.children:
            child_string = str(child).split("\n")
            child_string = "".join(["    "+string+"\n" for string in child_string])
            return_string += f"{child_string}"
        return_string = return_string[:-1] + "\n)"
        return return_string
    
    def to_tei(self):
        tei_string = ""
        for child in self.children:
            tei_string += child.to_tei()
        return f'<div type="book">{tei_string}</div>'
        

In [5]:
edition_directory = "./KuiSCIMA/optical_symbolic_dataset/05_shanghai_manuscript/"
book = Book([
    Juan(
        information_file=edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json",
        children=[
            Section(
                information_file=edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json",
                children=[
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/01_shangdiming.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/02_hezhibiao.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/03_huaihaizhuo.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/04_yuanzhishang.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/05_huangweichang.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/06_shushansui.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/07_shiyupei.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/08_wangzhongshan.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/09_dazairen.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/10_ougegui.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/11_fagongji.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/12_dilinyong.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/13_weisiye.json"),
                    Song(edition_directory + "./json/01_shengsongnaogeguchuiqushisishou/14_yanjingfu.json"),
                ]),
            Section(
                information_file=edition_directory + "./json/02_qinquyishou/qinquyishou.json",
                children=[
                    #Description("./json/02_qinquyishou/ceshangdiao.json"),
                    #Table("./json/02_qinquyishou/diaoxianfa.json"),
                    Song(edition_directory + "./json/02_qinquyishou/015_guyuan.json"),
                ])
        ]),
    Juan(
        information_file=edition_directory + "./json/03_yuejiuge/yuejiuge.json",
        children=[
            Section(
                information_file=edition_directory + "./json/03_yuejiuge/yuejiuge.json",
                children=[
                    SongOfYue(edition_directory + "./json/03_yuejiuge/016_dishunchudiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/017_wangyuwudiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/018_yuewangyuediao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/019_yuexiangceshangdiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/020_xiangwanggupingdiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/021_taozhishenshuangdiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/022_caoeshucediao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/023_pangjiangjungaopingdiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/024_jingzhongzhongguanshangdiao.json"),
                    SongOfYue(edition_directory + "./json/03_yuejiuge/025_caixiaozizhongguanbanzhandiao.json"),
                    #Table("./json/03_yuejiuge/gujinpufa.json"),
                    #Description("./json/03_yuejiuge/zhezifa.json"),
                ]),
        ]),
    Juan(
        information_file=edition_directory + "./json/04_ling/ling.json",
        children=[
            Section(
                information_file=edition_directory + "./json/04_ling/ling.json",
                children=[
                    Song(edition_directory + "./json/04_ling/026_xiaochongshanling.json"),
                    Song(edition_directory + "./json/04_ling/027_jiangmeiyin.json"),
                    Song(edition_directory + "./json/04_ling/028_moshanxi.json"),
                    Song(edition_directory + "./json/04_ling/029_yingshengraohonglou.json"),
                    Song(edition_directory + "./json/04_ling/030_geximeiling.json"),
                    Song(edition_directory + "./json/04_ling/031_ruanlanggui.json"),
                    Song(edition_directory + "./json/04_ling/032_ruanlanggui.json"),
                    Song(edition_directory + "./json/04_ling/033_haoshijin.json"),
                    Song(edition_directory + "./json/04_ling/034_dianjiangchun.json"),
                    Song(edition_directory + "./json/04_ling/035_dianjiangchun.json"),
                    Song(edition_directory + "./json/04_ling/036_yumeiren.json"),
                    Song(edition_directory + "./json/04_ling/037_yumeiren.json"),
                    Song(edition_directory + "./json/04_ling/038_yiwangsun.json"),
                    Song(edition_directory + "./json/04_ling/039_shaonianyou.json"),
                    Song(edition_directory + "./json/04_ling/040_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/041_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/042_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/043_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/044_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/045_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/046_zhegutian.json"),
                    Song(edition_directory + "./json/04_ling/047_yexingchuan.json"),
                    Song(edition_directory + "./json/04_ling/048_xinghuatianying.json"),
                    Song(edition_directory + "./json/04_ling/049_zuiyinshangxiaopin.json"),
                    Song(edition_directory + "./json/04_ling/050_yumeiling.json"),
                    Song(edition_directory + "./json/04_ling/051_tashaxing.json"),
                    Song(edition_directory + "./json/04_ling/052_suzhongqing.json"),
                    Song(edition_directory + "./json/04_ling/053_huanxisha.json"),
                    Song(edition_directory + "./json/04_ling/054_huanxisha.json"),
                    Song(edition_directory + "./json/04_ling/055_huanxisha.json"),
                    Song(edition_directory + "./json/04_ling/056_huanxisha.json"),
                    Song(edition_directory + "./json/04_ling/057_huanxisha.json"),
                    Song(edition_directory + "./json/04_ling/058_huanxisha.json"),
                ]),
        ]),
    Juan(
        information_file=edition_directory + "./json/05_man/man.json",
        children=[
            Section(
                information_file=edition_directory + "./json/05_man/man.json",
                children=[
                    Song(edition_directory + "./json/05_man/059_nishangzhongxu.json"),
                    Song(edition_directory + "./json/05_man/060_qinggongchun.json"),
                    Song(edition_directory + "./json/05_man/061_qitianle.json"),
                    Song(edition_directory + "./json/05_man/062_manjianghong.json"),
                    Song(edition_directory + "./json/05_man/063_yiehong.json"),
                    Song(edition_directory + "./json/05_man/064_niannujiao.json"),
                    Song(edition_directory + "./json/05_man/065_niannujiao.json"),
                    Song(edition_directory + "./json/05_man/066_meiwu.json"),
                    Song(edition_directory + "./json/05_man/067_yuexiadi.json"),
                    Song(edition_directory + "./json/05_man/068_qingboyin.json"),
                    Song(edition_directory + "./json/05_man/069_faquxianxianyin.json"),
                    Song(edition_directory + "./json/05_man/070_pipaxian.json"),
                    Song(edition_directory + "./json/05_man/071_linglongsifan.json"),
                    Song(edition_directory + "./json/05_man/072_cefan.json"),
                    Song(edition_directory + "./json/05_man/073_shuilongyin.json"),
                    Song(edition_directory + "./json/05_man/074_tanchunman.json"),
                    Song(edition_directory + "./json/05_man/075_bagui.json"),
                    Song(edition_directory + "./json/05_man/076_jielianhuan.json"),
                    Song(edition_directory + "./json/05_man/077_xiyingqianman.json"),
                    Song(edition_directory + "./json/05_man/078_moyuer.json"),
                ]),
        ]),
    Juan(
        information_file=edition_directory + "./json/06_ziduqu/ziduqu.json",
        children=[
            Section(
                information_file=edition_directory + "./json/06_ziduqu/ziduqu.json",
                children=[
                    Song(edition_directory + "./json/06_ziduqu/079_yangzhouman.json"),
                    Song(edition_directory + "./json/06_ziduqu/080_changtingyuanman.json"),
                    Song(edition_directory + "./json/06_ziduqu/081_danhuangliu.json"),
                    Song(edition_directory + "./json/06_ziduqu/082_shihuxian.json"),
                    Song(edition_directory + "./json/06_ziduqu/083_anxiang.json"),
                    Song(edition_directory + "./json/06_ziduqu/084_shuying.json"),
                    Song(edition_directory + "./json/06_ziduqu/085_xihongyi.json"),
                    Song(edition_directory + "./json/06_ziduqu/086_jueshao.json"),
                    Song(edition_directory + "./json/06_ziduqu/087_zhishao.json"),
                ]),
        ]),
    Juan(
        information_file=edition_directory + "./json/07_zizhiqu/zizhiqu.json",
        children=[
            Section(
                information_file=edition_directory + "./json/07_zizhiqu/zizhiqu.json",
                children=[
                    Song(edition_directory + "./json/07_zizhiqu/088_quixiaoyin.json"),
                    Song(edition_directory + "./json/07_zizhiqu/089_qiliangfan.json"),
                    Song(edition_directory + "./json/07_zizhiqu/090_cuilouyin.json"),
                    Song(edition_directory + "./json/07_zizhiqu/091_xiangyue.json"),
                    #Song(edition_directory + "./json/07_zizhiqu/qingyuanhuiyao.json"),
                ]),
        ]),
    Juan(
        information_file=edition_directory + "./json/08_bieji/bieji.json",
        children=[
            Section(
                information_file=edition_directory + "./json/08_bieji/bieji.json",
                children=[
                    Song(edition_directory + "./json/08_bieji/092_xiaochongshanling.json"),
                    Song(edition_directory + "./json/08_bieji/093_niannujiao.json"),
                    Song(edition_directory + "./json/08_bieji/094_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/095_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/096_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/097_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/098_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/099_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/100_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/101_busuanzi.json"),
                    Song(edition_directory + "./json/08_bieji/102_dongxiange.json"),
                    Song(edition_directory + "./json/08_bieji/103_moshanxi.json"),
                    Song(edition_directory + "./json/08_bieji/104_yongyule.json"),
                    Song(edition_directory + "./json/08_bieji/105_yumeiren.json"),
                    Song(edition_directory + "./json/08_bieji/106_yongyule.json"),
                    Song(edition_directory + "./json/08_bieji/107_shuidiaogetou.json"),
                    Song(edition_directory + "./json/08_bieji/108_hangongchun.json"),
                    Song(edition_directory + "./json/08_bieji/109_hangongchun.json"),
                ]),
        ]),])

In [7]:
print(book[1][0][5].to_tei())

<div type="song"><lg><lb/><notatedMusic type="lvlvpu"/><l>海門碧𠔃崔嵬潬上去𠔃潬下來予乗舟𠔃遲女目屢</l><lb/><notatedMusic type="lvlvpu"/><l>眩𠔃漚飛</l><lb/><notatedMusic type="lvlvpu"/><l>白馬駃𠔃素縿舞驅銀山𠔃疊萬鼓汨予從天𠔃南逝</l><lb/><notatedMusic type="lvlvpu"/><l>經西陵𠔃掠漁浦</l><lb/><notatedMusic type="lvlvpu"/><l>夫在舶𠔃婦在房風浩浩𠔃波茫茫𤁋予酒𠔃神龍府</l><lb/><notatedMusic type="lvlvpu"/><l>我征至𠔃無所苦</l></lg><p>右<title>濤之神雙調</title></p></div>


In [172]:
def transform_juan_to_tei(juan_path: str):
    tei_string = ""
    all_jsons = sorted(os.listdir(juan_path))
    
    pieces = [json for json in all_jsons if json[0].isnumeric()]
    others = [json for json in all_jsons if not json[0].isnumeric()]
    
    for other in others:
        tei_string += transform_juan_description_to_tei(os.path.join(juan, other))
            
    for piece in pieces:
        tei_string += transform_piece_to_tei(os.path.join(juan, piece))
            
    return f'<div type="juan">{tei_string}</div>'

In [ ]:
def transform_book_to_tei(juan_division: list[str], other: str):
    tei_string = ""
    
    # Table of Contents
    # Juan 1: 01_shengsongnaogeguchuiqushisishou, 02_qinquyishou
    tei_string 
    # Juan 2: 03_yuejiuge
    # Juan 3: 04_ling
    # Juan 4: 05_man
    # Juan 5: 06_ziduqu
    # Juan 6: 07_zizhiqu
    # Appendix: 08_bieji
        
    return f'<div type="book">{tei_string}</div>'
    #return f'<div type="book" style="writing-mode: vertical-rl">{tei_string}</div>'

In [208]:
base_directory = "./json"

all_files = []
for folder in sorted(next(os.walk(base_directory))[1]):
    everything = sorted(os.listdir(os.path.join(base_directory, folder)))
    everything = [os.path.join(base_directory, folder, e) for e in everything]
    pieces = [e for e in everything if e[0].isnumeric()]
    other = [e for e in everything if not e[0].isnumeric()]
    
    all_files += other
    all_files += pieces
    
print(all_files)


['./json/01_shengsongnaogeguchuiqushisishou/01_shangdiming.json', './json/01_shengsongnaogeguchuiqushisishou/02_hezhibiao.json', './json/01_shengsongnaogeguchuiqushisishou/03_huaihaizhuo.json', './json/01_shengsongnaogeguchuiqushisishou/04_yuanzhishang.json', './json/01_shengsongnaogeguchuiqushisishou/05_huangweichang.json', './json/01_shengsongnaogeguchuiqushisishou/06_shushansui.json', './json/01_shengsongnaogeguchuiqushisishou/07_shiyupei.json', './json/01_shengsongnaogeguchuiqushisishou/08_wangzhongshan.json', './json/01_shengsongnaogeguchuiqushisishou/09_dazairen.json', './json/01_shengsongnaogeguchuiqushisishou/10_ougegui.json', './json/01_shengsongnaogeguchuiqushisishou/11_fagongji.json', './json/01_shengsongnaogeguchuiqushisishou/12_dilinyong.json', './json/01_shengsongnaogeguchuiqushisishou/13_weisiye.json', './json/01_shengsongnaogeguchuiqushisishou/14_yanjingfu.json', './json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json', './json/02_qinquyishou/015